# cet-perch — audio cache builder

Reads the curated training-set list (`training_set_composition.csv`) and the
master metadata parquet, loads raw audio for every kept window, resamples to
32 kHz, pads/crops to exactly 160 000 samples (5 s), and writes:

- `X_audio.npy`       — `(N, 160000)` float32, mmap-friendly row-major
- `meta_train.parquet` — aligned metadata with `audio_row`, `group_key`,
  `dataset`, `coarse_class` columns

These two files are the sole inputs required by `cet_perch_finetune_run.ipynb`.

---

## Label mapping

- If `label_t2 == 'background'`  → `coarse_class = 'background'`
- Otherwise                      → `coarse_class = label_t4` (species level)

## Audio loading

All datasets here use **Pattern A** — on-the-fly load from WAV:
```
librosa.load(wav_path, sr=None, offset=offset_s, duration=5.0, mono=True)
→ soxr.resample(audio, native_sr, 32000)
→ pad / crop to 160000 samples
```
Datasets that require special handling (FREMANTLE, MONISH, Watkins pickled
windows) are handled in a separate notebook.


## 0. Imports

In [1]:
import os, warnings, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import librosa
import soxr
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED      = 42
TARGET_SR = 32_000
AUDIO_LEN = 160_000   # 5 s @ 32 kHz

np.random.seed(SEED)
print("Imports OK")

Imports OK


In [25]:
# Show everything from meta_all where dataset == "adriatic sea"

import pandas as pd

meta_all = pd.read_parquet(
    "/data2/mromaniuc/cet-det/alltogether/full_exploration/dim_red/projector_input/meta_all.parquet"
)

# inspect unique dataset names first (useful because capitalization/spaces may differ)
print(sorted(meta_all["dataset"].dropna().unique()))

# filter
adriatic = meta_all[
    (meta_all["dataset"].astype(str).str.lower() == "oltremare") &
    (meta_all["label_t1"].astype(str).str.lower() == "mammal")
]

print("\nRows found:", len(adriatic))

# show all columns + rows
with pd.option_context(
    "display.max_columns", None,
    "display.max_rows", 50,
    "display.max_colwidth", None,
    "display.width", 200
):
    display(adriatic)

['ALNITAK_CAVANILLES', 'Adriatic_Sea', 'DCLDE_2026', 'DOLPHINFREE', 'DRYAD', 'ECOSS_annot', 'ECOSS_enhanced', 'ECOSS_testtrain', 'FREMANTLE', 'MONISH', 'OLTREMARE', 'WATKINS']

Rows found: 2904


,wav_path,offset_s,label,n_clicks,click_count,n_whistles,whistle_subclasses,embedding_idx,dataset,environment,region,global_idx,row,wav_name,deployment_folder,deployment_id,species_folder,n_species,species_set,signal_set,deployment_key,deploymentID,deploymentPlatform,bottomDepth (m),label_refined,soundfile,duration_s,labels,label_str,provider,label_counts,ecotypes,labels_str,ecotypes_str,label_counts_str,label_clean,subdataset,split,n_whistle,n_bout,n_noise,file,source,src_path,segment,native_sr,spec_path,n_echolocation,n_burst_pulse,n_feeding_buzz,label_raw,species,source_file,window_idx,window_kind,original_duration_s,sig_start_sample,sig_len_sample,label_t1,label_t2,label_t3,label_t4,label_t5
228282,/data2/mromaniuc/cet-det/datasets/OLTREMARE/RAW_RECORDINGS/20211120_102109_192.wav,30.0,echolocation,NaN,NaN,0.0,None,6.0,OLTREMARE,pool,med,6,228282,None,None,None,None,NaN,None,None,None,None,None,NaN,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,0.0,None,None,None,NaN,NaN,None,2.0,0.0,0.0,echolocation,None,None,NaN,None,NaN,NaN,NaN,mammal,odontocete,click,Tursiops_truncatus,Tursiops_truncatus_click
228283,/data2/mromaniuc/cet-det/datasets/OLTREMARE/RAW_RECORDINGS/20211120_102109_192.wav,35.0,echolocation,NaN,NaN,0.0,None,7.0,OLTREMARE,pool,med,7,228283,None,None,None,None,NaN,None,None,None,None,None,NaN,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,0.0,None,None,None,NaN,NaN,None,1.0,0.0,0.0,echolocation,None,None,NaN,None,NaN,NaN,NaN,mammal,odontocete,click,Tursiops_truncatus,Tursiops_truncatus_click
228285,/data2/mromaniuc/cet-det/datasets/OLTREMARE/RAW_RECORDINGS/20211120_102109_192.wav,45.0,echolocation,NaN,NaN,0.0,None,9.0,OLTREMARE,pool,med,9,228285,None,None,None,None,NaN,None,None,None,None,None,NaN,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,0.0,None,None,None,NaN,NaN,None,1.0,0.0,0.0,echolocation,None,None,NaN,None,NaN,NaN,NaN,mammal,odontocete,click,Tursiops_truncatus,Tursiops_truncatus_click
228286,/data2/mromaniuc/cet-det/datasets/OLTREMARE/RAW_RECORDINGS/20211120_102109_192.wav,50.0,echolocation,NaN,NaN,1.0,None,10.0,OLTREMARE,pool,med,10,228286,None,None,None,None,NaN,None,None,None,None,None,NaN,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,0.0,None,None,None,NaN,NaN,None,2.0,0.0,0.0,echolocation+whistle,None,None,NaN,None,NaN,NaN,NaN,mammal,odontocete,click,Tursiops_truncatus,Tursiops_truncatus_click
228287,/data2/mromaniuc/cet-det/datasets/OLTREMARE/RAW_RECORDINGS/20211120_102109_192.wav,55.0,echolocation,NaN,NaN,0.0,None,11.0,OLTREMARE,pool,med,11,228287,None,None,None,None,NaN,None,None,None,None,None,NaN,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,0.0,None,None,None,NaN,NaN,None,3.0,0.0,0.0,echolocation,None,None,NaN,None,NaN,NaN,NaN,mammal,odontocete,click,Tursiops_truncatus,Tursiops_truncatus_click
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245035,/data2/mromaniuc/cet-det/datasets/OLTREMARE/RAW_RECORDINGS/20211121_100607_192.wav,95.0,whistle,NaN,NaN,1.0,None,16759.0,OLTREMARE,pool,med,16759,245035,None,None,None,None,NaN,None,None,None,None,None,NaN,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,0.0,None,None,None,NaN,NaN,None,0.0,0.0,0.0,whistle,None,None,NaN,None,NaN,NaN,NaN,mammal,odontocete,whistle,Tursiops_truncatus,Tursiops_truncatus_whistle
245036,/data2/mromaniuc/cet-det/datasets/OLTREMARE/RAW_RECORDINGS/20211121_100607_192.wav,100.0,whistle,NaN,NaN,4.0,None,16760.0,OLTREMARE,pool,med,16760,245036,None,None,None,None,NaN,None,None,None,None,None,NaN,None,None,NaN,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,0.0,None,None,None,NaN,NaN,None,0.0,0.0,0.0,whistle,None,None,NaN,None,NaN,N

## 1. Paths — edit these to match your filesystem

In [5]:
# ── Inputs ────────────────────────────────────────────────────────────────────

# Master metadata (all datasets, all windows)
META_ALL_PATH = Path('/data2/mromaniuc/cet-det/alltogether/full_exploration/dim_red/projector_input/meta_all.parquet')

# Curated keep-list produced by the curation notebook
CURATION_CSV  = Path('/data2/mromaniuc/cet-det/cet_perchv2/training_set_composition.csv')

# ── Outputs ───────────────────────────────────────────────────────────────────
OUT_DIR       = Path('/data2/mromaniuc/cet-det/cet_perchv2')
AUDIO_CACHE   = OUT_DIR / 'X_audio.npy'
META_OUT      = OUT_DIR / 'meta_train.parquet'

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset root folders ──────────────────────────────────────────────────────
# Add / remove entries as needed. Keys must match the 'dataset' column in meta_all.
# Values are the root folder where WAV files for that dataset live.
# wav_path in meta_all is already absolute for most datasets — this dict is used
# as a fallback if wav_path doesn't exist (e.g. mount-point changed).
DATASET_ROOTS = {
    'adriatic sea'   : Path('/data2/mromaniuc/cet-det/datasets/ADRIATIC_SEA/Audio_data'),
    'ALNITAK CAVANILLES'        : Path('/data2/mromaniuc/cet-det/datasets/ALNITAK_CAVANILLES'),
    'dclde_2026'          : Path('/data2/mromaniuc/cet-det/datasets/DCLDE_2026/audio'),
    'DOLPHINFREE'    : Path('/data2/mromaniuc/cet-det/datasets/DOLPHINFREE/Single_hydrophone/inputs/Audio_data'),
    'DRYAD'    : Path('/data2/mromaniuc/cet-det/datasets/DRYAD/Audio'),
    'ecoss_db_testing_training_sounds_metadata'          : Path('/data2/mromaniuc/cet-det/datasets/ECOSS/testingtraining_sounds'),
    'ecoss_db_annotated_sounds_metadata'          : Path('/data2/mromaniuc/cet-det/datasets/ECOSS/annotated_sounds'),
    'ecoss_db_enhanced4AI_sounds_metadata'          : Path('/data2/mromaniuc/cet-det/datasets/ECOSS/enhanced4AI_sounds'),
    'oltremare'      : Path('/data2/mromaniuc/cet-det/datasets/OLTREMARE/RAW_RECORDINGS'),
    # Pattern-B datasets (pickled windows) — handled separately, listed here
    # only so the presence check below doesn't error.
    'FREMANTLE'      : None,
    'MONISH'         : None,
    'archived-watkins'        : None,
}

# ── Sanity check ──────────────────────────────────────────────────────────────
print(f"META_ALL_PATH : {'OK' if META_ALL_PATH.exists() else 'MISSING'}  {META_ALL_PATH}")
print(f"CURATION_CSV  : {'OK' if CURATION_CSV.exists()  else 'MISSING'}  {CURATION_CSV}")
print(f"OUT_DIR       : {OUT_DIR}")
print()
for ds, root in DATASET_ROOTS.items():
    if root is None:
        print(f"  {ds:20s}  Pattern-B (skip here)")
    else:
        status = 'OK' if root.exists() else 'MISSING'
        print(f"  {ds:20s}  {status}  {root}")

META_ALL_PATH : OK  /data2/mromaniuc/cet-det/alltogether/full_exploration/dim_red/projector_input/meta_all.parquet
CURATION_CSV  : OK  /data2/mromaniuc/cet-det/cet_perchv2/training_set_composition.csv
OUT_DIR       : /data2/mromaniuc/cet-det/cet_perchv2

  adriatic sea          OK  /data2/mromaniuc/cet-det/datasets/ADRIATIC_SEA/Audio_data
  ALNITAK CAVANILLES    OK  /data2/mromaniuc/cet-det/datasets/ALNITAK_CAVANILLES
  dclde_2026            OK  /data2/mromaniuc/cet-det/datasets/DCLDE_2026/audio
  DOLPHINFREE           OK  /data2/mromaniuc/cet-det/datasets/DOLPHINFREE/Single_hydrophone/inputs/Audio_data
  DRYAD                 OK  /data2/mromaniuc/cet-det/datasets/DRYAD/Audio
  ecoss_db_testing_training_sounds_metadata  OK  /data2/mromaniuc/cet-det/datasets/ECOSS/testingtraining_sounds
  ecoss_db_annotated_sounds_metadata  OK  /data2/mromaniuc/cet-det/datasets/ECOSS/annotated_sounds
  ecoss_db_enhanced4AI_sounds_metadata  OK  /data2/mromaniuc/cet-det/datasets/ECOSS/enhanced4AI_sounds
 

## 2. Load metadata and curation list

In [ ]:
print("Loading master metadata...")
meta_all = pd.read_parquet(META_ALL_PATH)
print(f"  meta_all: {len(meta_all):,} rows")

print("Loading curation list...")
curation = pd.read_csv(CURATION_CSV)
print(f"  curation: {len(curation):,} rows")

# Drop rows marked for dropping
curation_keep = curation[curation['action'] != 'DROP'].copy()
print(f"  Non-dropped entries: {len(curation_keep)}")

# ── Map fine_label to meta_all ────────────────────────────────────────────────
# meta_all uses label_t4 for species and label_t2 for background.
# The curation CSV 'fine_label' column corresponds to the fine-grained label
# used during curation — we match against meta_all['label'] which is the
# original raw label.
# Adjust FINE_LABEL_COL if your meta_all uses a different column.
FINE_LABEL_COL = 'label'   # column in meta_all that matches curation 'fine_label'
DATASET_COL    = 'dataset' # column in meta_all that matches curation 'dataset'

assert FINE_LABEL_COL in meta_all.columns, \
    f"Expected '{FINE_LABEL_COL}' in meta_all. Available: {list(meta_all.columns)}"
assert DATASET_COL in meta_all.columns, \
    f"Expected '{DATASET_COL}' in meta_all. Available: {list(meta_all.columns)}"

# ── Subsample meta_all per (dataset, fine_label) group ───────────────────────
kept_frames = []

for _, spec in curation_keep.iterrows():
    ds        = spec['dataset']
    fine_lbl  = spec['fine_label']
    n_kept    = int(spec['n_kept'])
    action    = spec['action']

    # Match rows in meta_all
    mask = (meta_all[DATASET_COL] == ds) & (meta_all[FINE_LABEL_COL] == fine_lbl)
    group = meta_all[mask].copy()

    if len(group) == 0:
        print(f"  WARNING: no rows found for dataset='{ds}' fine_label='{fine_lbl}'")
        continue

    if action == 'KEEP_ALL' or len(group) <= n_kept:
        kept_frames.append(group)
    else:
        # Subsample deterministically
        kept_frames.append(group.sample(n=n_kept, random_state=SEED))

meta_kept = pd.concat(kept_frames, ignore_index=True)
print(f"\n  Total kept windows: {len(meta_kept):,}")
print(f"\nPer coarse_class summary:")

# Attach coarse_class from curation CSV for reporting
curation_map = curation_keep.set_index(['dataset','fine_label'])['coarse_class'].to_dict()
meta_kept['coarse_class_from_curation'] = meta_kept.apply(
    lambda r: curation_map.get((r[DATASET_COL], r[FINE_LABEL_COL]), None), axis=1
)
for cls, grp in meta_kept.groupby('coarse_class_from_curation'):
    print(f"  {cls:40s} {len(grp):>6,}")

Loading master metadata...
  meta_all: 247,630 rows
Loading curation list...
  curation: 67 rows
  Non-dropped entries: 64


ValueError: No objects to concatenate

## 3. Derive `coarse_class` and `group_key`

In [ ]:
# ── coarse_class ──────────────────────────────────────────────────────────────
# Rule: background → label_t2; species → label_t4
def derive_coarse_class(row):
    if str(row.get('label_t2', '')).lower() == 'background':
        return 'background'
    val = row.get('label_t4', None)
    if pd.isna(val) or val is None or str(val).strip() == '':
        # Fallback: try label_t3 then label_t2
        for col in ['label_t3', 'label_t2']:
            v = row.get(col, None)
            if pd.notna(v) and str(v).strip() != '':
                return str(v).strip()
        return 'unknown'
    return str(val).strip()

meta_kept['coarse_class'] = meta_kept.apply(derive_coarse_class, axis=1)

print("Coarse class distribution:")
for cls, n in meta_kept['coarse_class'].value_counts().items():
    print(f"  {cls:40s} {n:>6,}")

unknown_count = (meta_kept['coarse_class'] == 'unknown').sum()
if unknown_count > 0:
    print(f"\n  WARNING: {unknown_count} rows with coarse_class='unknown' — inspect these")
    print(meta_kept[meta_kept['coarse_class'] == 'unknown'][['global_idx','dataset','label_t2','label_t3','label_t4']].head(10))

In [ ]:
# ── group_key ─────────────────────────────────────────────────────────────────
# group_key must uniquely identify a recording file or deployment so that
# GroupShuffleSplit keeps all windows from the same file in the same split.
#
# Priority:
#   1. deployment_key (if present and not null)
#   2. wav_path (absolute path identifies the file)
#   3. dataset + wav_name

def derive_group_key(row):
    dk = row.get('deployment_key', None)
    if pd.notna(dk) and str(dk).strip() != '':
        return str(dk).strip()
    wp = row.get('wav_path', None)
    if pd.notna(wp) and str(wp).strip() != '':
        return str(wp).strip()
    wn = row.get('wav_name', None)
    if pd.notna(wn) and str(wn).strip() != '':
        return f"{row.get('dataset','')}_{wn}"
    return f"{row.get('dataset','')}_{row.get('global_idx','')}"

meta_kept['group_key'] = meta_kept.apply(derive_group_key, axis=1)

n_groups = meta_kept['group_key'].nunique()
print(f"Unique group_keys: {n_groups:,}  (out of {len(meta_kept):,} windows)")
print(f"Mean windows per group: {len(meta_kept)/n_groups:.1f}")

# Per-dataset group count
print("\nGroups per dataset:")
for ds, grp in meta_kept.groupby('dataset'):
    print(f"  {ds:20s}  {grp['group_key'].nunique():>5,} groups  {len(grp):>6,} windows")

## 4. Identify Pattern-B datasets and filter to Pattern-A only

Pattern-B datasets (FREMANTLE, MONISH, Watkins) have pre-cached pickled windows
and are handled in a separate notebook. This notebook only materialises
Pattern-A (on-the-fly WAV loading).

In [ ]:
PATTERN_B_DATASETS = {'FREMANTLE', 'MONISH', 'Watkins'}

meta_a = meta_kept[~meta_kept['dataset'].isin(PATTERN_B_DATASETS)].copy()
meta_b = meta_kept[ meta_kept['dataset'].isin(PATTERN_B_DATASETS)].copy()

print(f"Pattern-A (WAV loading):  {len(meta_a):,} windows across {meta_a['dataset'].nunique()} datasets")
print(f"Pattern-B (pickle cache): {len(meta_b):,} windows across {meta_b['dataset'].nunique()} datasets")
print()
print("Pattern-A datasets:")
for ds, grp in meta_a.groupby('dataset'):
    print(f"  {ds:20s}  {len(grp):>6,} windows")
if len(meta_b) > 0:
    print("\nPattern-B datasets (skipped here):")
    for ds, grp in meta_b.groupby('dataset'):
        print(f"  {ds:20s}  {len(grp):>6,} windows")

## 5. Resolve WAV paths

For each row, check that `wav_path` exists. If not, try to reconstruct it
from `DATASET_ROOTS`.

In [ ]:
def resolve_wav_path(row):
    """Return a resolved Path if the file exists, else None."""
    p = row.get('wav_path', None)
    if pd.notna(p) and Path(p).exists():
        return str(p)
    # Try reconstructing from DATASET_ROOTS + wav_name or soundfile
    root = DATASET_ROOTS.get(row.get('dataset', ''), None)
    if root is not None:
        for name_col in ['wav_name', 'soundfile', 'source_file', 'file']:
            fname = row.get(name_col, None)
            if pd.notna(fname) and fname:
                candidate = root / str(fname)
                if candidate.exists():
                    return str(candidate)
    return None

print("Resolving WAV paths...")
meta_a['resolved_wav'] = meta_a.apply(resolve_wav_path, axis=1)

n_missing = meta_a['resolved_wav'].isna().sum()
n_ok      = meta_a['resolved_wav'].notna().sum()
print(f"  Resolved : {n_ok:,}")
print(f"  Missing  : {n_missing:,}")

if n_missing > 0:
    print("\n  Missing WAV samples (first 20):")
    missing = meta_a[meta_a['resolved_wav'].isna()]
    print(missing[['global_idx','dataset','wav_path']].head(20).to_string())
    print("\n  Fix DATASET_ROOTS above or check wav_path values before continuing.")

## 6. Audio loading helpers

In [ ]:
def load_window(wav_path, offset_s, target_sr=TARGET_SR, target_len=AUDIO_LEN):
    """
    Load a 5-second window from a WAV file.

    Steps:
      1. librosa.load at native sample rate (no resampling yet)
      2. soxr.resample to target_sr (high-quality)
      3. Pad with zeros if shorter than target_len
      4. Crop to target_len if longer

    Returns float32 array of shape (target_len,).
    """
    audio_native, native_sr = librosa.load(
        wav_path,
        sr=None,                  # load at native SR
        offset=float(offset_s),
        duration=5.0,
        mono=True,
    )
    if native_sr != target_sr:
        audio = soxr.resample(audio_native, native_sr, target_sr)
    else:
        audio = audio_native

    # Pad / crop
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    audio = audio[:target_len]

    return audio.astype(np.float32)


def smoke_test_loader(meta_df, n_samples=5):
    """Load a handful of windows and check shapes / amplitude ranges."""
    sample = meta_df[meta_df['resolved_wav'].notna()].sample(
        min(n_samples, len(meta_df)), random_state=SEED
    )
    print("Smoke-testing audio loader:")
    for _, row in sample.iterrows():
        try:
            audio = load_window(row['resolved_wav'], row['offset_s'])
            print(f"  [{row['dataset']:15s}] shape={audio.shape}  "
                  f"min={audio.min():.4f}  max={audio.max():.4f}  "
                  f"rms={np.sqrt(np.mean(audio**2)):.4f}  "
                  f"coarse_class={row['coarse_class']}")
        except Exception as e:
            print(f"  [{row['dataset']:15s}] ERROR: {e}")

smoke_test_loader(meta_a)

## 7. Materialise audio cache

Allocates `X_audio.npy` on disk as a memory-mapped array and fills it row by
row. If the run is interrupted, just re-run this cell — it will skip rows
that are already non-zero (you can disable this with `RESUME = False`).

In [ ]:
RESUME = True   # set False to overwrite from scratch

# Only process rows with a resolved WAV path
meta_process = meta_a[meta_a['resolved_wav'].notna()].copy().reset_index(drop=True)
N = len(meta_process)
print(f"Windows to materialise: {N:,}")

# Allocate or open existing mmap
if AUDIO_CACHE.exists() and RESUME:
    print(f"Opening existing cache (RESUME=True): {AUDIO_CACHE}")
    X_audio = np.load(str(AUDIO_CACHE), mmap_mode='r+')
    if X_audio.shape != (N, AUDIO_LEN):
        print(f"  WARNING: shape mismatch {X_audio.shape} vs expected ({N}, {AUDIO_LEN})")
        print("  Set RESUME=False to recreate from scratch.")
else:
    print(f"Allocating new cache: ({N}, {AUDIO_LEN}) float32  "
          f"= {N * AUDIO_LEN * 4 / 1e9:.2f} GB")
    X_audio = np.lib.format.open_memmap(
        str(AUDIO_CACHE), mode='w+', dtype=np.float32, shape=(N, AUDIO_LEN)
    )

# Fill row by row
errors = []
for i, row in tqdm(meta_process.iterrows(), total=N, desc='Loading audio'):
    if RESUME and X_audio[i].any():   # skip already-filled rows
        continue
    try:
        X_audio[i] = load_window(row['resolved_wav'], row['offset_s'])
    except Exception as e:
        errors.append((i, row['global_idx'], row['dataset'], str(e)))
        X_audio[i] = np.zeros(AUDIO_LEN, dtype=np.float32)  # fill with silence

X_audio.flush()
print(f"\nDone. {N - len(errors):,} windows loaded, {len(errors)} errors.")

if errors:
    print("\nErrors (first 20):")
    for i, gidx, ds, msg in errors[:20]:
        print(f"  row={i}  global_idx={gidx}  dataset={ds}  error={msg}")

## 8. Validate the cache

In [ ]:
print("Validating cache...")
X_audio_r = np.load(str(AUDIO_CACHE), mmap_mode='r')
print(f"  Shape  : {X_audio_r.shape}")
print(f"  Dtype  : {X_audio_r.dtype}")
print(f"  Global min/max: {X_audio_r.min():.4f} / {X_audio_r.max():.4f}")

# Check per-class amplitude distributions
print("\nPer-class amplitude stats (RMS):")
for cls in sorted(meta_process['coarse_class'].unique()):
    idx = meta_process[meta_process['coarse_class'] == cls].index.tolist()
    sample_idx = idx[:min(50, len(idx))]
    rms_vals = np.sqrt(np.mean(X_audio_r[sample_idx] ** 2, axis=1))
    print(f"  {cls:40s}  n={len(idx):>5,}  "
          f"rms_mean={rms_vals.mean():.5f}  rms_std={rms_vals.std():.5f}")

# Check for all-zero rows (failed loads)
zero_rows = np.where(~X_audio_r.any(axis=1))[0]
print(f"\nAll-zero rows (failed loads): {len(zero_rows)}")
if len(zero_rows) > 0:
    print(f"  Row indices: {zero_rows[:20]}")

## 9. Write `meta_train.parquet`

In [ ]:
# audio_row must equal the row position in X_audio
meta_process = meta_process.reset_index(drop=True)
meta_process['audio_row'] = meta_process.index.astype(np.int64)

# Columns required by the fine-tuning notebook
required_cols = ['audio_row', 'group_key', 'dataset', 'coarse_class']

# Additional useful columns to carry forward
extra_cols = [
    'global_idx', 'wav_path', 'offset_s',
    'label_t2', 'label_t3', 'label_t4',
    'label', 'region', 'environment',
]
extra_cols = [c for c in extra_cols if c in meta_process.columns]

meta_out = meta_process[required_cols + extra_cols].copy()

# Assertions
assert (meta_out['audio_row'].values == np.arange(len(meta_out))).all(), \
    "audio_row must equal row position"
assert meta_out['group_key'].notna().all(), "group_key has nulls"
assert meta_out['coarse_class'].notna().all(), "coarse_class has nulls"
assert len(meta_out) == X_audio_r.shape[0], \
    f"meta_out length {len(meta_out)} != X_audio rows {X_audio_r.shape[0]}"

meta_out.to_parquet(META_OUT, index=False)
print(f"Written: {META_OUT}")
print(f"  Rows    : {len(meta_out):,}")
print(f"  Columns : {list(meta_out.columns)}")
print()
print("Final class distribution:")
for cls, n in meta_out['coarse_class'].value_counts().items():
    print(f"  {cls:40s} {n:>6,}")
print()
print("Final dataset distribution:")
for ds, n in meta_out['dataset'].value_counts().items():
    print(f"  {ds:25s} {n:>6,}")

## 10. Final compatibility check with fine-tuning notebook

In [ ]:
print("Final compatibility check...")

X_check   = np.load(str(AUDIO_CACHE), mmap_mode='r')
meta_check = pd.read_parquet(META_OUT)

assert X_check.shape[1] == AUDIO_LEN, \
    f"X_audio width {X_check.shape[1]} != {AUDIO_LEN}"
assert X_check.dtype == np.float32, \
    f"X_audio dtype {X_check.dtype} != float32"
assert len(meta_check) == X_check.shape[0], \
    f"Length mismatch: meta {len(meta_check)} vs X_audio {X_check.shape[0]}"
assert (meta_check['audio_row'].values == np.arange(len(meta_check))).all(), \
    "audio_row not aligned with row position"
assert set(required_cols).issubset(meta_check.columns), \
    f"Missing required columns: {set(required_cols) - set(meta_check.columns)}"

print("  X_audio.npy          OK")
print("  meta_train.parquet   OK")
print(f"  N windows            {X_check.shape[0]:,}")
print(f"  N classes            {meta_check['coarse_class'].nunique()}")
print(f"  N datasets           {meta_check['dataset'].nunique()}")
print(f"  N groups             {meta_check['group_key'].nunique():,}")
print()
print("Ready for cet_perch_finetune_run.ipynb  ✓")

## Notes

**Pattern-B datasets (FREMANTLE, MONISH, Watkins)** are not included here.
Run the Pattern-B cache notebook to produce separate arrays, then concatenate:

```python
X_a = np.load('X_audio_patternA.npy', mmap_mode='r')
X_b = np.load('X_audio_patternB.npy', mmap_mode='r')
X_all = np.concatenate([X_a, X_b], axis=0)
np.save('X_audio.npy', X_all)
# Then concatenate and re-index meta parquets accordingly.
```

**Resuming interrupted runs**: set `RESUME = True` (default) in cell 7.
Already-filled rows (non-zero) are skipped.

**Amplitude normalisation**: windows are NOT peak-normalised here.
Perch V2's PCEN frontend is robust to level variation — normalising would
remove real SNR information. If you want normalisation add it in the augmentation notebook.
